In [9]:
import sys
sys.path.append("..")
import torch
import json
from pathlib import Path
from tqdm import tqdm
from itertools import cycle
from datasets import load_from_disk
from torch.utils.data import DataLoader, TensorDataset

from sae_lens import SAE
from src.data import PROJECT_ROOT, RuleTakerDataset
from src.llm_upgrade import wrap_for_transformer_lens
from src.probing import collect_activations

# Параметры построения SAE

In [2]:
EXP_ID = "exp3-1"
MODEL_NAME = "EleutherAI/pythia-1b-deduped"
ADAPTER_PATH = PROJECT_ROOT / "results/checkpoints/finetune/exp3/checkpoint-2200"
VARIANT = "depth-0"
LAYER_IDX = 14
HOOK_POINT = f"blocks.{LAYER_IDX}.hook_resid_post"

In [11]:
D_MODEL = 2048
EXPANSION_FACTOR = 8
K = 32
LR = 2e-4
BATCH_SIZE_SAE = 256
STEPS = 2000
DEVICE = "cuda"

In [6]:
SAVE_DIR = PROJECT_ROOT / f"results/sae/{EXP_ID}_saelens"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
ACTS_SAVE_PATH = SAVE_DIR / "acts_layer14.pt"

# Загрузка модели, датасета и активаций

Если активации не собраны (не сохранены), то они собираются

In [7]:
if ACTS_SAVE_PATH.exists():
    print(f"Загрузка активаций: {ACTS_SAVE_PATH}")
    acts_tensor = torch.load(ACTS_SAVE_PATH, map_location="cpu").float()
else:
    print("Сбор активаций")
    hooked_model, tokenizer = wrap_for_transformer_lens(MODEL_NAME, str(ADAPTER_PATH), device=DEVICE)
    hooked_model.eval()

    cache_path = PROJECT_ROOT / "data" / "processed" / f"ruletaker_{VARIANT}_small"
    dataset = load_from_disk(str(cache_path))["train"]
    train_dataset = RuleTakerDataset(dataset, tokenizer, max_length=256)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

    acts_np, _ = collect_activations(
        hooked_model, train_loader, LAYER_IDX,
        hook_name="resid_post", pooling="last", device=DEVICE,
        save_path=str(ACTS_SAVE_PATH)
    )
    acts_tensor = torch.from_numpy(acts_np).float()
    del hooked_model
    torch.cuda.empty_cache()

Загрузка активаций: C:\MyPythonProjects\mephi_diss\results\sae\exp3-1_saelens\acts_layer14.pt


In [8]:
# Нормализация для стабильной сходимости
act_mean = acts_tensor.mean(dim=0)
act_std = acts_tensor.std(dim=0) + 1e-8
acts_tensor = (acts_tensor - act_mean) / act_std

In [12]:
act_dataset = TensorDataset(acts_tensor)
act_loader = cycle(DataLoader(act_dataset, batch_size=BATCH_SIZE_SAE, shuffle=True, drop_last=True))

# SAE

In [ ]:
sae = SAE({
    "architecture": "standard",
    "d_in": D_MODEL,
    "d_sae": D_MODEL * EXPANSION_FACTOR,
    "activation_fn_str": "topk",
    "activation_fn_kwargs": {"k": K},
    "apply_b_dec_to_input": False,
    "finetuning_scaling_factor": False,
    "context_size": 1,
    "model_name": MODEL_NAME,
    "hook_name": f"blocks.{LAYER_IDX}.hook_resid_post",
    "prepend_bos": True,
    "normalize_activations": "none",
    "dtype": torch.float32,
    "device": DEVICE,
}).to(DEVICE)

TypeError: Can't instantiate abstract class SAE with abstract methods decode, encode